In [2]:
import pandas as pd
import json

# 读取数据
df_examples = pd.read_parquet('../data/esci-data/shopping_queries_dataset_examples.parquet')
df_products = pd.read_parquet('../data/esci-data/shopping_queries_dataset_products.parquet')
df_sources = pd.read_csv("../data/esci-data/shopping_queries_dataset_sources.csv")


In [9]:
df_examples_products = pd.merge(
    df_examples,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)

df_task_1 = df_examples_products[df_examples_products['small_version'] == 1]

lang='es'
df_task_1_us = df_task_1[(df_task_1['product_locale'] == lang)]

df_task_1_train_us = df_task_1[(df_task_1["split"] == "train") & (df_task_1['product_locale'] == lang)]

df_task_1_test_us = df_task_1[(df_task_1["split"] == "test") & (df_task_1['product_locale'] == lang)]

In [ ]:
import json


selected_columns = df_task_1_us[['product_id', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color']]


product_info_dict = selected_columns.set_index('product_id').T.to_dict()

indexed_product_info_dict = {}
product_id_to_index = {}


for index, (product_id, product_info) in enumerate(product_info_dict.items()):
    indexed_product_info_dict[index] = product_info
    product_id_to_index[product_id] = index

print(len(indexed_product_info_dict))
print(len(product_id_to_index))






/var/folders/80/yk28jv5n41x_7lhz5w2ykd2h0000gn/T/ipykernel_52049/2264891886.py:7: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  product_info_dict = selected_columns.set_index('product_id').T.to_dict()


167761
167761


In [11]:

# 将字典保存为JSON文件
with open(f'../data/esci_{lang}/esci_{lang}.item.json', 'w', encoding='utf-8') as json_file:
    json.dump(indexed_product_info_dict, json_file, ensure_ascii=False, indent=4)

with open(f'../data/esci_{lang}/product_id_to_index.json', 'w') as f:
    json.dump(product_id_to_index, f)
print("JSON文件已成功保存为product_info.json")


JSON文件已成功保存为product_info.json


In [12]:
from collections import defaultdict
from tqdm import tqdm
train_query_docs = df_task_1_train_us[['query', 'product_id', 'esci_label']].drop_duplicates()
test_query_docs = df_task_1_test_us[['query', 'product_id', 'esci_label']].drop_duplicates()
# 转换为字典格式
train_data_dict = train_query_docs.groupby('query').apply(
    lambda x: list(zip(x['product_id'], x['esci_label']))
).to_dict()
test_data_dict = test_query_docs.groupby('query').apply(
    lambda x: list(zip(x['product_id'], x['esci_label']))
).to_dict()

# 导出为JSON文件
with open(f'../data/esci_{lang}/esci_{lang}.train.json', 'w') as json_file:
    json.dump(train_data_dict, json_file, ensure_ascii=False, indent=4)
with open(f'../data/esci_{lang}/esci_{lang}.test.json', 'w') as json_file:
    json.dump(test_data_dict, json_file, ensure_ascii=False, indent=4)
print("JSON文件已成功创建！")


/var/folders/80/yk28jv5n41x_7lhz5w2ykd2h0000gn/T/ipykernel_52049/2929981312.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_data_dict = train_query_docs.groupby('query').apply(
/var/folders/80/yk28jv5n41x_7lhz5w2ykd2h0000gn/T/ipykernel_52049/2929981312.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_data_dict = test_query_docs.groupby('query').apply(


JSON文件已成功创建！


In [ ]:
from collections import defaultdict
from tqdm import tqdm


pairs = []
for query in train_data_dict:
    for (product_id, esci_label) in train_data_dict[query]:
        if esci_label != 'I':
            pairs.append((query, product_id_to_index[product_id]))


query_to_docs = defaultdict(set)
for query, doc in pairs:
    query_to_docs[query].add(doc)


doc_to_queries = defaultdict(set)
for query, doc in pairs:
    doc_to_queries[doc].add(query)


total_queries = len(query_to_docs)
total_docs_per_query = sum(len(docs) for docs in query_to_docs.values())
avg_docs_per_query = total_docs_per_query / total_queries


total_docs = len(doc_to_queries)
total_queries_per_doc = sum(len(queries) for queries in doc_to_queries.values())
avg_queries_per_doc = total_queries_per_doc / total_docs

doc_to_relevance_docs = defaultdict(set)
for query in tqdm(query_to_docs):
    for doc in query_to_docs[query]:
        doc_to_relevance_docs[doc] = query_to_docs[query] - {doc}
    
doc_to_relevance_docs_serializable = {doc: list(relevance_docs) for doc, relevance_docs in doc_to_relevance_docs.items()}

with open(f'../data/esci_{lang}/doc_to_relevance_docs.json', 'w') as f:
    json.dump(doc_to_relevance_docs_serializable, f)


100%|██████████| 5632/5632 [00:00<00:00, 28635.06it/s]


In [14]:
from collections import defaultdict
from tqdm import tqdm
import json


train_query_docs = df_task_1_train_us[['query', 'product_id', 'esci_label']].drop_duplicates()
test_query_docs = df_task_1_test_us[['query', 'product_id', 'esci_label']].drop_duplicates()

train_id = set()
for index, row in train_query_docs.iterrows():
    if row['esci_label'] != 'I':
        train_id.add(row['product_id'])


new_test_rows = []


for index, row in test_query_docs.iterrows():
    if row['product_id'] in train_id and row['esci_label'] != 'I':
        new_test_rows.append(row)
print(len(new_test_rows))

new_test_query_docs = pd.DataFrame(new_test_rows, columns=['query', 'product_id', 'esci_label'])


test_data_seen_dict = new_test_query_docs.groupby('query').apply(
    lambda x: list(zip(x['product_id'], x['esci_label']))
).to_dict()
with open(f'../data/esci_{lang}/esci_{lang}.test.seen.json', 'w') as json_file:
    json.dump(test_data_seen_dict, json_file, ensure_ascii=False, indent=4)
print("JSON文件已成功创建！")


14763
JSON文件已成功创建！


/var/folders/80/yk28jv5n41x_7lhz5w2ykd2h0000gn/T/ipykernel_52049/595191772.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_data_seen_dict = new_test_query_docs.groupby('query').apply(
